
# Case B Hα/Hβ ratio across ionization and metallicity

In the dust-free limit with Case B recombination (T_e=10,000 K, n_e=100 cm^-3),
the intrinsic Hα/Hβ ratio is 2.86, nearly independent of ionization parameter
and metallicity below ~0.5 Z☉ (Storey & Hummer 1995, MNRAS 272, 41). This
diagnostic checks that tengri's Cue nebular emulator reproduces the canonical
value across its (logU, logZ_gas) grid, identifying any library drift or
implementation errors.

We build a young star-forming model with bare-stellar SSP, disable dust and
photon escape, then compute the intrinsic line ratio for all combinations
of logU ∈ [−4, −2] and logZ_gas ∈ [−2, 0.5].


In [ ]:
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# Load bare-stellar SSP (Cue requirement)
ssp = tengri.load_ssp("fsps_prsc_miles_chabrier")

# Build young SF model: logU and logZ_gas free, all else fixed to young/dust-free defaults
model = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "dpl",
        "all_params": tengri.FIXED,
        "alpha": 5.0,  # steep early assembly
        "beta": 2.0,
        "tau_gyr": 1.0,
        "log_total_mass": 10.0,
    },
    dust={"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0},
    neb={
        "type": "cue",
        "all_params": tengri.FIXED,
        "neb_logU": tengri.Uniform(-4.0, -2.0),
        "neb_logZ_gas": tengri.Uniform(-2.0, 0.5),
        "neb_fesc": 0.0,
        "neb_fesc_lya": 0.0,
    },
    redshift=tengri.Fixed(0.0),
)

baseline = dict(model.spec.sample(jax.random.PRNGKey(0)))

# Grid over logU and logZ_gas
log_u_vals = np.linspace(-4.0, -2.0, 6)
log_z_vals = np.linspace(-2.0, 0.5, 6)

# Compute Hα/Hβ ratio for each (logU, logZ_gas) pair
ratio_grid = np.zeros((len(log_u_vals), len(log_z_vals)))
for i, log_u in enumerate(log_u_vals):
    for j, log_z in enumerate(log_z_vals):
        params = {
            **baseline,
            "neb_logU": jnp.float64(log_u),
            "neb_logZ_gas": jnp.float64(log_z),
        }
        lines = model.predict(params).lines
        h_alpha = float(lines.halpha)
        h_beta = float(lines.hbeta)
        ratio_grid[i, j] = h_alpha / h_beta if h_beta > 0 else np.nan

# Plot 2D contour of Hα/Hβ with Case B reference
fig, ax = plt.subplots(figsize=(6.5, 4.5))

# Contourf with 15 levels
levels = np.linspace(np.nanmin(ratio_grid), np.nanmax(ratio_grid), 15)
cf = ax.contourf(log_z_vals, log_u_vals, ratio_grid, levels=levels, cmap="RdYlBu_r")

# Overlay contour line at the Case B reference value
case_b_ratio = 2.86
cs = ax.contour(
    log_z_vals,
    log_u_vals,
    ratio_grid,
    levels=[case_b_ratio],
    colors="black",
    linewidths=2.0,
    linestyles="--",
)
ax.clabel(cs, inline=True, fontsize=9, fmt=f"Case B = {case_b_ratio:.2f}")

ax.set_xlabel(r"$\log_{10}(Z_{\mathrm{gas}}/Z_{\odot})$")
ax.set_ylabel(r"$\log_{10}(U)$")
cbar = fig.colorbar(cf, ax=ax, pad=0.01)
cbar.set_label(r"Intrinsic $H\alpha / H\beta$ ratio")

fig.tight_layout()
plt.savefig("plot_diag_case_b_balmer_ratio.png", dpi=150, bbox_inches="tight")